In [1]:
# PORTAL = "http://v3tv.live:80"
# MAC = "00:1A:79:C2:01:8B"
PORTAL = "http://line.smootvone.vip"
MAC = "00:1A:79:AB:E8:8C"

In [2]:
import requests
import urllib3
from pprint import pprint

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

TIMEOUT = 15

HEADERS = {
    "User-Agent": "Mozilla/5.0 (QtEmbedded; U; Linux; C)",
    "X-User-Agent": "Model: MAG250; Link: WiFi",
    "Referer": f"{PORTAL}/c/",
    "Cookie": f"mac={MAC}; stb_lang=en; timezone=UTC",
}

session = requests.Session()


def handshake():
    url = f"{PORTAL}/portal.php"

    params = {
        "type": "stb",
        "action": "handshake",
        "token": "",
        "JsHttpRequest": "1-xml",
    }

    r = session.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=TIMEOUT,
        verify=False,
    )

    r.raise_for_status()

    data = r.json()

    token = data["js"]["token"]
    random = data["js"].get("random")

    return token, random


def get_profile(token):
    url = f"{PORTAL}/portal.php"

    headers = HEADERS.copy()
    headers["Authorization"] = f"Bearer {token}"

    params = {
        "type": "stb",
        "action": "get_profile",
        "hd": "1",
        "ver": "ImageDescription:0.2.18-r23-pub-250;",
        "num_banks": "2",
        "sn": "1234567890AB",
        "stb_type": "MAG250",
        "client_type": "STB",
        "image_version": "218",
        "video_out": "hdmi",
        "device_id": "",
        "device_id2": "",
        "signature": "",
        "auth_second_step": "1",
        "hw_version": "1.7-BD-00",
        "not_valid_token": "0",
        "metrics": '{"mac":"%s"}' % MAC,
        "hw_version_2": "unknown",
        "timestamp": "0",
        "api_signature": "262",
        "prehash": "0",
        "JsHttpRequest": "1-xml",
    }

    r = session.get(
        url,
        headers=headers,
        params=params,
        timeout=TIMEOUT,
        verify=False,
    )

    r.raise_for_status()
    return r.json()


def get_account_info(token):
    url = f"{PORTAL}/portal.php"

    headers = HEADERS.copy()
    headers["Authorization"] = f"Bearer {token}"

    params = {
        "type": "account_info",
        "action": "get_main_info",
        "JsHttpRequest": "1-xml",
    }

    r = session.get(
        url,
        headers=headers,
        params=params,
        timeout=TIMEOUT,
        verify=False,
    )

    r.raise_for_status()
    return r.json()


def get_channels(token):
    url = f"{PORTAL}/portal.php"

    headers = HEADERS.copy()
    headers["Authorization"] = f"Bearer {token}"

    params = {
        "type": "itv",
        "action": "get_all_channels",
        "force_ch_link_check": "",
        "JsHttpRequest": "1-xml",
    }

    r = session.get(
        url,
        headers=headers,
        params=params,
        timeout=TIMEOUT,
        verify=False,
    )

    r.raise_for_status()
    return r.json()

In [3]:
def create_link(token, cmd):
    url = f"{PORTAL}/portal.php"

    headers = HEADERS.copy()
    headers["Authorization"] = f"Bearer {token}"

    params = {
        "type": "itv",
        "action": "create_link",
        "cmd": cmd,
        "series": "0",
        "forced_storage": "0",
        "disable_ad": "0",
        "download": "0",
        "force_ch_link_check": "0",
        "JsHttpRequest": "1-xml",
    }

    r = session.get(
        url,
        headers=headers,
        params=params,
        timeout=TIMEOUT,
        verify=False,
    )

    r.raise_for_status()
    return r.json()

In [4]:
try:
    print("[*] Performing handshake...")
    token, random = handshake()

    print(f"[+] Token acquired: {token[:16]}...")

    print("\n[*] Loading profile...")
    profile = get_profile(token)
    # pprint(profile)

    print("\n[*] Fetching account info...")
    account = get_account_info(token)
    # pprint(account)

    js = account.get("js", {})

    expiration = (
        js.get("end_date")
        or js.get("phone")
        or js.get("tariff_plan")
        or js.get("account_expiration")
    )

    print("\n=== Account Expiration ===")
    print(expiration)

    print("\n[*] Fetching channels...")
    channels = get_channels(token)

    channel_list = channels.get("js", {}).get("data", [])

    print(f"\n=== Total Channels: {len(channel_list)} ===")

except requests.HTTPError as e:
    print(f"HTTP error: {e}")

except Exception as e:
    print(f"Error: {e}")

[*] Performing handshake...
[+] Token acquired: 13B5E37118A6321C...

[*] Loading profile...

[*] Fetching account info...

=== Account Expiration ===
April 15, 2027, 7:00 am

[*] Fetching channels...

=== Total Channels: 11170 ===


In [8]:
key = "main event"

for ch in channel_list:
    name = ch.get("name", "")

    if key.lower() in name.lower():

        channel_id = ch.get("id")

        cmd = ch.get("cmd")

        link_data = create_link(token, cmd)

        stream_url = link_data.get("js", {}).get("cmd")

        print("\n------------------------")
        print("Name :", name)
        print("ID   :", channel_id)
        print("CMD  :", cmd)
        print("URL  :", stream_url)


------------------------
Name : ┃UK┃ SKY SPORTS MAIN EVENTS 4K
ID   : 1312008
CMD  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=1312008&extension=ts&play_token=MhexZSw1Sj
URL  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=&extension=ts&play_token=gGY3Wx35Fr

------------------------
Name : ┃UK┃ SKY SPORTS MAIN EVENT 50 FPS
ID   : 1311999
CMD  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=1311999&extension=ts&play_token=JY1br9sC5Y
URL  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=&extension=ts&play_token=bUIKpvIHON

------------------------
Name : ┃UK┃ SKY SPORTS MAIN EVENTS HEVC FHD
ID   : 1312033
CMD  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=1312033&extension=ts&play_token=6jbHoax6h6
URL  : ffmpeg http://line.smootvone.vip:80/play/live.php?mac=00:1A:79:AB:E8:8C&stream=&extension=ts&play_token=DHNuYiJcXD

